# Sola Face LoRA — Colab (live log + fallbacks)

1. Runtime → GPU → **Disconnect and delete runtime** first if you already failed
2. Run cells top → bottom
3. Upload `foocus_new/datasets/sola_face_kohya.zip`
4. If train fails, the red error **and full log** appear in the same cell — copy that here


In [ ]:
# @title 0) GPU
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Enable GPU"
print(torch.cuda.get_device_name(0))


In [ ]:
# @title 1) Install
import os, shutil
os.chdir("/content")
!pip -q install -U pip wheel setuptools
!pip -q install torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu121
shutil.rmtree("/content/sd-scripts", ignore_errors=True)
!git clone --depth 1 https://github.com/kohya-ss/sd-scripts.git /content/sd-scripts
os.chdir("/content/sd-scripts")
!pip -q install accelerate==0.31.0 transformers==4.41.2 diffusers==0.29.2 safetensors==0.4.3
!pip -q install ftfy einops opencv-python-headless toml voluptuous bitsandbytes==0.43.1
!pip -q install invisible-watermark open-clip-torch==2.24.0 timm
print("files", [f for f in os.listdir('.') if 'train' in f][:8])


In [ ]:
# @title 2) Upload dataset zip
import os, zipfile, shutil
from google.colab import files
DATA="/content/sola_data"
shutil.rmtree(DATA, ignore_errors=True); os.makedirs(DATA, exist_ok=True); os.chdir(DATA)
print("Upload sola_face_kohya.zip")
up=files.upload(); assert up
for name in up:
  if name.lower().endswith('.zip'):
    zipfile.ZipFile(name).extractall(DATA)
found=None
for r,ds,_ in os.walk(DATA):
  if '10_sola_face' in ds:
    found=os.path.join(r,'10_sola_face'); break
if not found:
  jpgs=[os.path.join(r,f) for r,_,fs in os.walk(DATA) for f in fs if f.lower().endswith(('.jpg','.jpeg','.png'))]
  assert len(jpgs)>=10, len(jpgs)
  found=os.path.join(DATA,'10_sola_face'); os.makedirs(found, exist_ok=True)
  for src in jpgs:
    stem=os.path.splitext(os.path.basename(src))[0]
    shutil.copy2(src, os.path.join(found, stem+'.jpg'))
    t=os.path.splitext(src)[0]+'.txt'; td=os.path.join(found, stem+'.txt')
    if os.path.isfile(t): shutil.copy2(t, td)
    else: open(td,'w').write('sola_face, photo of a woman, looking at camera\n')
TRAIN_ROOT=os.path.dirname(found)
OUT='/content/outputs/sola_face_lora'; os.makedirs(OUT, exist_ok=True)
n=len([f for f in os.listdir(found) if f.lower().endswith('.jpg')])
print('TRAIN_ROOT', TRAIN_ROOT, 'images', n); assert n>=10


In [ ]:
# @title 3) Train with LIVE log + automatic fallbacks
import os, sys, subprocess, glob, time

os.chdir('/content/sd-scripts')
LOG = OUT + '/train.log'
open(LOG,'w').close()

def run(cmd):
    print('\n===== RUNNING =====\n', ' '.join(cmd), flush=True)
    env=os.environ.copy()
    env['PYTHONPATH']='/content/sd-scripts' + os.pathsep + env.get('PYTHONPATH','')
    env['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'
    with open(LOG,'a',encoding='utf-8') as logf:
        logf.write('\n\nCMD ' + ' '.join(cmd) + '\n')
        logf.flush()
        p=subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env, cwd='/content/sd-scripts', bufsize=1)
        for line in p.stdout:
            print(line, end='')
            logf.write(line)
        return p.wait()

base = [
  sys.executable, 'sdxl_train_network.py',
  '--pretrained_model_name_or_path=stabilityai/stable-diffusion-xl-base-1.0',
  f'--train_data_dir={TRAIN_ROOT}',
  f'--output_dir={OUT}',
  '--output_name=sola_face_sdxl',
  '--save_model_as=safetensors',
  '--save_precision=fp16',
  '--caption_extension=.txt',
  '--enable_bucket',
  '--train_batch_size=1',
  '--gradient_checkpointing',
  '--max_train_epochs=8',
  '--save_every_n_epochs=2',
  '--learning_rate=1e-4',
  '--unet_lr=1e-4',
  '--text_encoder_lr=5e-5',
  '--lr_scheduler=cosine',
  '--lr_warmup_steps=20',
  '--network_module=networks.lora',
  '--mixed_precision=fp16',
  '--cache_latents',
  '--cache_latents_to_disk',
  '--seed=42',
  '--keep_tokens=1',
  '--max_data_loader_n_workers=0',
]

attempts = [
  base + ['--resolution=512,512','--min_bucket_reso=256','--max_bucket_reso=1024','--network_dim=8','--network_alpha=8','--optimizer_type=AdamW8bit'],
  base + ['--resolution=512,512','--min_bucket_reso=256','--max_bucket_reso=1024','--network_dim=8','--network_alpha=8','--optimizer_type=AdamW'],
  base + ['--resolution=448,448','--min_bucket_reso=256','--max_bucket_reso=768','--network_dim=4','--network_alpha=4','--optimizer_type=AdamW'],
]

ok=False
for i,cmd in enumerate(attempts,1):
    print(f'\n######## ATTEMPT {i}/{len(attempts)} ########', flush=True)
    code=run(cmd)
    paths=glob.glob(OUT+'/**/*.safetensors', recursive=True)
    print('exit', code, 'safetensors', paths, flush=True)
    if code==0 and paths:
        ok=True
        break
    # clear CUDA cache between attempts
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

if not ok:
    print('\n===== COPY THIS LOG TO CHAT =====\n')
    print(open(LOG,encoding='utf-8',errors='replace').read()[-12000:])
    raise RuntimeError('All train attempts failed. Paste the log block above into chat.')
else:
    print('SUCCESS', glob.glob(OUT+'/**/*.safetensors', recursive=True))


In [ ]:
# @title 4) Download LoRA
import glob, os
from google.colab import files
paths=sorted(glob.glob('/content/outputs/sola_face_lora/**/*.safetensors', recursive=True), key=os.path.getmtime)
print(paths)
assert paths
best=[p for p in paths if 'sola_face_sdxl' in os.path.basename(p)] or paths
files.download(best[-1])


In [ ]:
# @title DEBUG only — dump log if you already failed
!wc -l /content/outputs/sola_face_lora/train.log || true
!tail -n 200 /content/outputs/sola_face_lora/train.log || true
